# Figure 7 — GSE235046 data download

This notebook downloads the author-supplied raw count table for the source
mitoxyperiosis RNA-seq experiment from NCBI GEO and writes a locked sample
manifest for the 15 BMDM samples (9-hour treatment; n=3 per condition).

Run this notebook before `01_Figure7_Analysis.ipynb`.


In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(
    os.environ.get("MITOXY_PROJECT_ROOT", "/scratch/ar58064/mitoxy")
).resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "gse235046"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "figure_7_clean"
FIGURE_DIR = PROJECT_ROOT / "figures" / "figure_7_clean"
for directory in [RAW_DIR, OUTPUT_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)


Project root: /scratch/ar58064/mitoxy


In [2]:
import gzip
import hashlib
import json
import shutil
import urllib.request

ACCESSION = "GSE235046"
COUNT_URL = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE235nnn/"
    "GSE235046/suppl/GSE235046_Count_table.txt.gz"
)
COUNT_PATH = RAW_DIR / "GSE235046_Count_table.txt.gz"


def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_atomic(url, destination):
    destination = Path(destination)
    temporary = destination.with_suffix(destination.suffix + ".part")
    request = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 Figure7-reproducibility"},
    )
    with urllib.request.urlopen(request, timeout=180) as response:
        with temporary.open("wb") as output:
            shutil.copyfileobj(response, output)
    temporary.replace(destination)


if not COUNT_PATH.exists():
    print("Downloading", ACCESSION, "count table...")
    download_atomic(COUNT_URL, COUNT_PATH)
else:
    print("Using existing count table:", COUNT_PATH)

with gzip.open(COUNT_PATH, "rt") as handle:
    header = handle.readline().rstrip("\n").split("\t")
    first_data_row = handle.readline().rstrip("\n").split("\t")

if len(header) < 16 or len(first_data_row) != len(header):
    raise RuntimeError(
        "The downloaded count table does not have the expected tabular form."
    )

download_manifest = {
    "accession": ACCESSION,
    "url": COUNT_URL,
    "path": str(COUNT_PATH),
    "size_bytes": COUNT_PATH.stat().st_size,
    "sha256": sha256(COUNT_PATH),
    "n_columns": len(header),
}
(RAW_DIR / "download_manifest.json").write_text(
    json.dumps(download_manifest, indent=2), encoding="utf-8"
)

print(json.dumps(download_manifest, indent=2))
print("Count-table integrity check: PASS")


{
  "accession": "GSE235046",
  "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE235nnn/GSE235046/suppl/GSE235046_Count_table.txt.gz",
  "path": "/scratch/ar58064/mitoxy/data/raw/gse235046/GSE235046_Count_table.txt.gz",
  "size_bytes": 1110939,
  "sha256": "683ec07b3d81ea8cb6b834b08e071ba9ec2b89f49c61ac303029864b1a110906",
  "n_columns": 19
}
Count-table integrity check: PASS


## Locked sample manifest


In [3]:
sample_manifest = pd.DataFrame([
    ["GSM7493746", "BMDM_Media_01", "Media", 1, 0, 0, 0],
    ["GSM7493747", "BMDM_Media_02", "Media", 2, 0, 0, 0],
    ["GSM7493748", "BMDM_Media_03", "Media", 3, 0, 0, 0],
    ["GSM7493749", "BMDM_LPS_01", "LPS", 1, 1, 0, 0],
    ["GSM7493750", "BMDM_LPS_02", "LPS", 2, 1, 0, 0],
    ["GSM7493751", "BMDM_LPS_03", "LPS", 3, 1, 0, 0],
    ["GSM7493752", "BMDM_CS_01", "CS", 1, 0, 1, 0],
    ["GSM7493753", "BMDM_CS_02", "CS", 2, 0, 1, 0],
    ["GSM7493754", "BMDM_CS_03", "CS", 3, 0, 1, 0],
    ["GSM7493755", "BMDM_LPS+CS_01", "LPS+CS", 1, 1, 1, 0],
    ["GSM7493756", "BMDM_LPS+CS_02", "LPS+CS", 2, 1, 1, 0],
    ["GSM7493757", "BMDM_LPS+CS_03", "LPS+CS", 3, 1, 1, 0],
    ["GSM7493758", "BMDM_LPS+CS+Torin_01", "LPS+CS+Torin", 1, 1, 1, 1],
    ["GSM7493759", "BMDM_LPS+CS+Torin_02", "LPS+CS+Torin", 2, 1, 1, 1],
    ["GSM7493760", "BMDM_LPS+CS+Torin_03", "LPS+CS+Torin", 3, 1, 1, 1],
], columns=[
    "geo_accession", "sample_title", "condition", "replicate",
    "lps", "carbon_starvation", "torin",
])
sample_manifest["duration_hours"] = 9
sample_manifest.to_csv(RAW_DIR / "GSE235046_sample_manifest.csv", index=False)

assert len(sample_manifest) == 15
assert sample_manifest.groupby("condition").size().eq(3).all()
display(sample_manifest)
print("Sample-manifest validation: PASS")
print("Saved raw inputs to:", RAW_DIR)


,geo_accession,sample_title,condition,replicate,lps,carbon_starvation,torin,duration_hours
0,GSM7493746,BMDM_Media_01,Media,1,0,0,0,9
1,GSM7493747,BMDM_Media_02,Media,2,0,0,0,9
2,GSM7493748,BMDM_Media_03,Media,3,0,0,0,9
3,GSM7493749,BMDM_LPS_01,LPS,1,1,0,0,9
4,GSM7493750,BMDM_LPS_02,LPS,2,1,0,0,9
5,GSM7493751,BMDM_LPS_03,LPS,3,1,0,0,9
6,GSM7493752,BMDM_CS_01,CS,1,0,1,0,9
7,GSM7493753,BMDM_CS_02,CS,2,0,1,0,9
8,GSM7493754,BMDM_CS_03,CS,3,0,1,0,9
9,GSM7493755,BMDM_LPS+CS_01,LPS+CS,1,1,1,0,9


Sample-manifest validation: PASS
Saved raw inputs to: /scratch/ar58064/mitoxy/data/raw/gse235046
